# SmartPlate — SegFormer-B0 Full Training (Google Colab)

Run this notebook on Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).
Expected total time: ~35–45 minutes.

When training finishes the checkpoint is saved to your Google Drive at
`MyDrive/smartplate_checkpoints/segformer_best/`.
Download it and place it at `code/checkpoints/segformer_best/` in your local repo.

## 0. Colab setup

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate pandas tqdm pillow

In [ ]:
# Mount Google Drive — checkpoint will be saved here so it survives session end
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Clone the repo (skip if already done)
if not os.path.exists('/content/smartplate'):
    !git clone https://github.com/krishdembla/smartplate.git /content/smartplate

# Work from the notebooks directory so relative paths match the local setup
os.chdir('/content/smartplate/code/notebooks')
print('Working directory:', os.getcwd())

## 1. Config

In [ ]:
QUICK_SMOKE_TEST = False   # set True for a 5-min sanity check before the full run

IMAGE_SIZE    = 512
BATCH_SIZE    = 4          # T4 has 15 GB VRAM — batch 4 at 512x512 is comfortable
NUM_EPOCHS    = 5
LEARNING_RATE = 6e-5
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2          # Linux DataLoader workers are fine on Colab
EVAL_BATCH_SIZE = 8
SEED          = 42

BASE_CHECKPOINT = 'nvidia/segformer-b0-finetuned-ade-512-512'
DRIVE_SAVE_DIR  = '/content/drive/MyDrive/smartplate_checkpoints/segformer_best'
LOCAL_SAVE_DIR  = '../checkpoints/segformer_best'   # also write inside repo clone
HISTORY_PATH    = '../checkpoints/train_history.json'

if QUICK_SMOKE_TEST:
    NUM_EPOCHS  = 1
    SUBSET_SIZE = 200
    print('*** QUICK SMOKE TEST MODE ***')
else:
    SUBSET_SIZE = None

## 2. Environment

In [ ]:
import os
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import sys, json, math, time, random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from PIL import Image

sys.path.insert(0, os.path.abspath('..'))
from smartplate import load_class_to_group, ALL_GROUPS

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch  : {torch.__version__}')
print(f'Device : {device}')
if device == 'cpu':
    print('WARNING: no GPU detected — make sure Runtime → Change runtime type → T4 GPU is selected.')

## 3. Dataset + label mappings

In [ ]:
from datasets import load_dataset

ds = load_dataset('EduardoPacheco/FoodSeg103')
print(ds)

FOODSEG103_CLASSES = [
    'background', 'candy', 'egg tart', 'french fries', 'chocolate', 'biscuit', 'popcorn',
    'pudding', 'ice cream', 'cheese butter', 'cake', 'wine', 'milkshake', 'coffee', 'juice',
    'milk', 'tea', 'almond', 'red beans', 'cashew', 'dried cranberries', 'soy', 'walnut',
    'peanut', 'egg', 'apple', 'date', 'apricot', 'avocado', 'banana', 'strawberry', 'cherry',
    'blueberry', 'raspberry', 'mango', 'olives', 'peach', 'lemon', 'pear', 'fig', 'pineapple',
    'grape', 'kiwi', 'melon', 'orange', 'watermelon', 'steak', 'pork', 'chicken duck',
    'sausage', 'fried meat', 'lamb', 'sauce', 'crab', 'fish', 'shellfish', 'shrimp', 'soup',
    'bread', 'corn', 'hamburg', 'pizza', 'hanamaki baozi', 'wonton dumplings', 'pasta',
    'noodles', 'rice', 'pie', 'tofu', 'eggplant', 'potato', 'garlic', 'cauliflower', 'tomato',
    'kelp', 'seaweed', 'spring onion', 'rape', 'ginger', 'okra', 'lettuce', 'pumpkin',
    'cucumber', 'white radish', 'carrot', 'asparagus', 'bamboo shoots', 'broccoli',
    'celery stick', 'cilantro mint', 'snow peas', 'cabbage', 'bean sprouts', 'onion',
    'pepper', 'green beans', 'French beans', 'king oyster mushroom', 'shiitake',
    'enoki mushroom', 'oyster mushroom', 'white button mushroom', 'salad', 'other ingredients',
]
NUM_LABELS = len(FOODSEG103_CLASSES)  # 104
id2label = {i: n for i, n in enumerate(FOODSEG103_CLASSES)}
label2id = {n: i for i, n in enumerate(FOODSEG103_CLASSES)}

class_to_group = load_class_to_group('../assets/foodseg103_to_groups.csv')
print(f'{NUM_LABELS} classes, {len(class_to_group)} mapped to food groups.')

if SUBSET_SIZE is not None:
    ds['train']      = ds['train'].shuffle(seed=SEED).select(range(SUBSET_SIZE))
    ds['validation'] = ds['validation'].shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(ds['validation']))))
    print(f'Subset: train={len(ds["train"])}  val={len(ds["validation"])}')

## 4. Model + processor

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

processor = SegformerImageProcessor(
    do_resize=True,
    size={'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    do_reduce_labels=False,
)

model = SegformerForSemanticSegmentation.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'SegFormer-B0 trainable params: {n_params:.2f} M')

## 5. Transforms + DataLoaders

In [ ]:
def train_transform(batch):
    images, labels = [], []
    for img, lbl in zip(batch['image'], batch['label']):
        img = img.convert('RGB')
        lbl_np = np.array(lbl)
        if random.random() < 0.5:
            img    = img.transpose(Image.FLIP_LEFT_RIGHT)
            lbl_np = np.ascontiguousarray(lbl_np[:, ::-1])
        images.append(img)
        labels.append(Image.fromarray(lbl_np))
    return processor(images=images, segmentation_maps=labels, return_tensors='pt')

def val_transform(batch):
    images = [x.convert('RGB') for x in batch['image']]
    labels = [Image.fromarray(np.array(x)) for x in batch['label']]
    return processor(images=images, segmentation_maps=labels, return_tensors='pt')

train_ds = ds['train'].with_transform(train_transform)
val_ds   = ds['validation'].with_transform(val_transform)

def collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'].squeeze(0) for b in batch]),
        'labels':       torch.stack([b['labels'].squeeze(0) for b in batch]).long(),
    }

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val  : {len(val_ds)} samples, {len(val_loader)} batches')

## 6. Evaluation helpers

In [ ]:
from tqdm.auto import tqdm

GROUP_NAMES = ('background',) + ALL_GROUPS
GROUP_TO_ID = {g: i for i, g in enumerate(GROUP_NAMES)}

class_to_group_id = np.zeros(NUM_LABELS, dtype=np.int64)
for cid, gname in class_to_group.items():
    class_to_group_id[cid] = GROUP_TO_ID[gname]

def _iou_from_confmat(conf):
    tp = np.diag(conf).astype(np.float64)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    denom = tp + fp + fn
    return np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)

def evaluate_model(model, loader, device):
    model.eval()
    conf = np.zeros((NUM_LABELS, NUM_LABELS), dtype=np.int64)
    running_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc='eval', leave=False):
            pixel_values = batch['pixel_values'].to(device)
            labels       = batch['labels'].to(device)
            out = model(pixel_values=pixel_values, labels=labels)
            running_loss += out.loss.item(); n_batches += 1
            logits = torch.nn.functional.interpolate(
                out.logits, size=labels.shape[-2:], mode='bilinear', align_corners=False,
            )
            preds = logits.argmax(dim=1).cpu().numpy().ravel()
            gt    = labels.cpu().numpy().ravel()
            conf += np.bincount(
                gt * NUM_LABELS + preds,
                minlength=NUM_LABELS * NUM_LABELS,
            ).reshape(NUM_LABELS, NUM_LABELS)
    per_class_iou = _iou_from_confmat(conf)
    miou_classes  = float(np.nanmean(per_class_iou))
    n_groups = len(GROUP_NAMES)
    group_conf = np.zeros((n_groups, n_groups), dtype=np.int64)
    for gt_cls in range(NUM_LABELS):
        gt_g = class_to_group_id[gt_cls]
        for pr_cls in range(NUM_LABELS):
            group_conf[gt_g, class_to_group_id[pr_cls]] += conf[gt_cls, pr_cls]
    per_group_iou = _iou_from_confmat(group_conf)
    miou_groups   = float(np.nanmean(per_group_iou))
    pixel_acc = float(np.diag(conf).sum() / max(conf.sum(), 1))
    return {
        'val_loss':      running_loss / max(n_batches, 1),
        'miou_classes':  miou_classes,
        'miou_groups':   miou_groups,
        'pixel_acc':     pixel_acc,
        'per_class_iou': per_class_iou,
        'per_group_iou': per_group_iou,
    }

## 7. Training loop

In [ ]:
import shutil

Path(LOCAL_SAVE_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_SAVE_DIR).mkdir(parents=True, exist_ok=True)
Path(HISTORY_PATH).parent.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

history = []
best_miou = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'epoch {epoch}/{NUM_EPOCHS}')
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        labels       = batch['labels'].to(device)
        out = model(pixel_values=pixel_values, labels=labels)
        optimizer.zero_grad(set_to_none=True)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item(); n_batches += 1
        if n_batches % 50 == 0:
            pbar.set_postfix(loss=f'{running_loss/n_batches:.3f}')
    train_loss = running_loss / max(n_batches, 1)
    eval_stats = evaluate_model(model, val_loader, device)
    dt = time.time() - t0

    row = {
        'epoch':        epoch,
        'train_loss':   train_loss,
        'val_loss':     eval_stats['val_loss'],
        'miou_classes': eval_stats['miou_classes'],
        'miou_groups':  eval_stats['miou_groups'],
        'pixel_acc':    eval_stats['pixel_acc'],
        'seconds':      dt,
    }
    history.append(row)
    print(
        f'epoch {epoch}: train_loss={train_loss:.3f}  val_loss={eval_stats["val_loss"]:.3f}  '
        f'mIoU_103={eval_stats["miou_classes"]:.3f}  mIoU_7grp={eval_stats["miou_groups"]:.3f}  '
        f'pixelAcc={eval_stats["pixel_acc"]:.3f}  ({dt/60:.1f} min)'
    )

    if eval_stats['miou_classes'] > best_miou:
        best_miou = eval_stats['miou_classes']
        # Save locally inside the repo clone
        model.save_pretrained(LOCAL_SAVE_DIR)
        processor.save_pretrained(LOCAL_SAVE_DIR)
        np.save(Path(LOCAL_SAVE_DIR) / 'per_class_iou.npy', eval_stats['per_class_iou'])
        np.save(Path(LOCAL_SAVE_DIR) / 'per_group_iou.npy', eval_stats['per_group_iou'])
        # Mirror to Drive so it survives session disconnects
        shutil.copytree(LOCAL_SAVE_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
        print(f'  ↳ saved new best to Drive (mIoU_103={best_miou:.3f})')

    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f, indent=2)

print('\nTraining complete.')

## 8. Final evaluation

In [ ]:
import pandas as pd
from transformers import SegformerForSemanticSegmentation

best_model = SegformerForSemanticSegmentation.from_pretrained(LOCAL_SAVE_DIR).to(device)
final = evaluate_model(best_model, val_loader, device)
print(f'FINAL  mIoU_103={final["miou_classes"]:.3f}  mIoU_7grp={final["miou_groups"]:.3f}  pixelAcc={final["pixel_acc"]:.3f}')

pc = pd.DataFrame({
    'class_id':   np.arange(NUM_LABELS),
    'class_name': FOODSEG103_CLASSES,
    'iou':        final['per_class_iou'],
}).sort_values('iou', ascending=False)
print('\n--- top 15 classes by IoU ---')
print(pc.head(15).to_string(index=False))

pg = pd.DataFrame({'group': list(GROUP_NAMES), 'iou': final['per_group_iou']})
print('\n--- per food group ---')
print(pg.to_string(index=False))

# Save CSVs to Drive alongside the checkpoint
pc.to_csv(Path(DRIVE_SAVE_DIR) / 'per_class_iou.csv', index=False)
pg.to_csv(Path(DRIVE_SAVE_DIR) / 'per_group_iou.csv', index=False)
print(f'\nResults CSVs saved to {DRIVE_SAVE_DIR}')

## 9. Download checkpoint

Your checkpoint is already saved to Google Drive at `MyDrive/smartplate_checkpoints/segformer_best/`.

To bring it back to your local machine:
1. Open Google Drive → find `smartplate_checkpoints/segformer_best/`
2. Download as ZIP, unzip into `code/checkpoints/segformer_best/` in your local repo
3. Run the cell below to also zip it here for a direct Colab download

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/segformer_best.zip'
shutil.make_archive('/content/segformer_best', 'zip', LOCAL_SAVE_DIR)
print(f'Zipped to {zip_path} — downloading...')
files.download(zip_path)